# Hillstrom: gửi email nữ cho ai? Uplift modeling và CATE

Module 03 đã cho thấy email nữ (Womens E-Mail) làm tỉ lệ ghé website tăng khoảng 4.5 điểm phần trăm
**trung bình**. Nhưng email tốn tiền và làm phiền khách; câu hỏi kinh doanh là gửi cho **ai**. Mục
tiêu: ước lượng conditional average treatment effect (CATE) $$\tau(x) = E[Y(1) - Y(0) \mid X = x]$$
cho từng khách, xếp hạng khách theo uplift, và đánh giá bảng xếp hạng đó một cách trung thực.

Vì Hillstrom là thí nghiệm ngẫu nhiên, ta có thể **đánh giá** mô hình uplift trên tập test: đó là
điều không làm được với dữ liệu quan sát.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` là chỗ phân tích tự do; phần lớn điểm nằm ở đó và ở `REPORT.md`.
- Chấm: `uv run pytest tracks/data-science/modules/06-causal-inference/exercises/advanced/02-hillstrom-uplift -q`

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# explore: đọc dữ liệu thật (tải bằng: uv run python data/download_dataset.py hillstrom)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
raw = pd.read_csv(ROOT / "data/raw/hillstrom/hillstrom.csv")
raw.segment.value_counts()

## 1. Chuẩn bị

Giữ hai nhánh `arm` và `control`, thêm cột `treatment` (1 cho `arm`), và one-hot các cột phân loại
(`zip_code`, `channel`; `history_segment` trùng thông tin với `history` nên bỏ). Trả về DataFrame
cùng index với dữ liệu gốc.

In [ ]:
def prepare(df: pd.DataFrame, arm: str = "Womens E-Mail", control: str = "No E-Mail") -> pd.DataFrame:
    # TODO: lọc hai nhánh, treatment, pd.get_dummies(..., columns=["zip_code", "channel"], dtype=float)
    raise NotImplementedError

## 2. T-learner

Huấn luyện một mô hình phân loại trên nhóm treated, một trên nhóm control (cùng một "nhà máy" mô
hình `make_model()`), dự đoán uplift của mỗi khách trên tập `score` bằng
$$\hat\tau(x) = \hat\mu_1(x) - \hat\mu_0(x)$$ (hiệu hai xác suất `predict_proba[:, 1]`).

In [ ]:
def t_learner(train: pd.DataFrame, score: pd.DataFrame, features: list[str], outcome: str,
              make_model, treatment: str = "treatment") -> np.ndarray:
    # TODO
    raise NotImplementedError

## 3. Đánh giá: Qini curve

Sắp khách theo uplift dự đoán giảm dần (hoà thì giữ thứ tự gốc: `np.argsort(-uplift, kind="stable")`).
Với k khách đầu, gọi $$N_t, N_c$$ là số khách treated / control và $$Y_t, Y_c$$ là số khách có
outcome = 1 trong từng nhóm. Qini:

$$
Q(k) = Y_t(k) - Y_c(k)\,\frac{N_t(k)}{N_c(k)}
$$

(khi $$N_c(k) = 0$$ thì $$Q(k) = Y_t(k)$$.)

$$Q(k)$$ là "số outcome tăng thêm nếu chỉ gửi cho k khách đầu" (quy về cỡ nhóm treated). Đường
ngẫu nhiên là $$\frac{k}{n} Q(n)$$. Qini coefficient:

$$
\text{qini} = \frac{1}{n} \cdot \frac{1}{n} \sum_{k=1}^{n} \left( Q(k) - \frac{k}{n} Q(n) \right)
$$

In [ ]:
def qini_curve(uplift: np.ndarray, treatment: np.ndarray, outcome: np.ndarray) -> pd.DataFrame:
    """DataFrame n dòng, cột frac (k / n) và qini (Q(k)), k = 1..n."""
    # TODO
    raise NotImplementedError


def qini_coefficient(uplift: np.ndarray, treatment: np.ndarray, outcome: np.ndarray) -> float:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: TODO chia train/test ngẫu nhiên (seed cố định), T-learner với ít nhất hai loại mô hình,
# Qini trên test, so với xếp hạng đơn giản (ví dụ cột womens), uplift theo nhóm (top 20/40/...%)

## 4. Quy ra tiền

Email tốn $0.10, biên lợi nhuận 30%, nhưng bài này dùng `visit` làm outcome. Hãy đề xuất cách nối
uplift trên visit với lợi nhuận (hoặc chuyển sang `spend`), chọn ngưỡng gửi, và ước tính lợi nhuận
tăng thêm trên 100 000 khách kèm khoảng không chắc chắn.

In [ ]:
# explore: TODO